In [1]:
import ROOT as r
import fedrarootlogon 
from matplotlib import pyplot as plt
import matplotlib
import awkward as ak
import uproot
import numpy as np
import sys
sys.path.insert(0, "/home/vincenzo/Documents/Vincenzo_FOOT/")
from Clustering_Cosmici_Frammenti import *
import time


def g_func(x, p1, p2, p3):
    return p1* np.exp(- (x - p2)**2 / (2*p3**2) )
brick_id = "GSI1"

Welcome to JupyROOT 6.24/06
Load FEDRA libs


In [2]:
%jsroot on 

In [3]:
track_name = "b000111.2.0.0.trk.root"
file_name = "b11_vol.root"

### Calcolo delle variabili di volume e scrittura del Tree con i branch aggiuntivi

In [4]:
#Calcolo_Variabili_Volume_New(file_name, track_name)

### Tagli iniziali sul dataset: k0 >= 1, nessun taglio su ki (i>0) e nessun taglio sul rapporto ki+1/ki

In [5]:
k0_min = 1

+ " && s[0].X()<=96250 && s[0].X()>=28750 && s[0].Y()<=86000 && s[0].Y() >=14000

In [6]:
file = r.TFile(file_name, "READ")

tracks_V = file.Get("tracks_n")

file2 = r.TFile("b11_cuts.root", "RECREATE")

t_tracks_V = tracks_V.CopyTree("k0>=" + str(k0_min) + " && VR0_av < 25000")
t_tracks_V.Write("tracks_k0_k1")
file2.Close()
file.Close()

In [7]:
k_stat = 1

In [8]:
file2 = r.TFile("b11_cuts.root", "READ")

tracks_2 = file2.Get("tracks_k0_k1")

c = r.TCanvas()
tracks_2.Draw("VR0_av:tan>>vr0_tan(80, 0,1, 80, 0, 21000)", "", "COLZ")

vr0_tan = r.gDirectory.Get("vr0_tan")
vr0_tan.SetTitle("VR0_{av} vs tan(#theta); tan(#theta); VR0_{av}")
vr0_tan.Draw("COLZ")

c.Draw()

In [9]:
tracks_2.Draw("k0", "npl >= 33")
c.Draw()

In [10]:
c = r.TCanvas()
tracks_2.Draw("VR0_av>>vr0", "", "COLZ")

vr0_tan = r.gDirectory.Get("vr0")
vr0_tan.SetTitle("VR0_{av} [k_{0}>0]; VR0_{av}; Entries")
vr0_tan.Draw("COLZ")

t1 = r.TText(20000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")


c.Draw()

## Taglio Cosmici/Frammenti

### Distribuzione con tutte le tracce -> definizione delle curve per il taglio

In [11]:
a0, b0 = 2600, 0.6
a1, b1 = 3850, 0.7
a2, b2 = (a1+a0)/2, 0.65

a2 = 3537.5

c = r.TCanvas()
tracks_2.Draw("VR0_av:tan>>vr0_tan(80, 0,1, 80, 0, 21000)", "k1<2 && k2<2 && k3<2", "COLZ")

vr0_tan = r.gDirectory.Get("vr0_tan")
vr0_tan.SetTitle("VR0_{av} vs tan(#theta) [k_{1}<2 & k_{2}<2 & k_{3}<2]; tan(#theta); VR0_{av}")
vr0_tan.Draw("COLZ")
cut_curve = r.TF1("cut", str(a0)+"*(1 + exp("+str(b0)+ " *x*x))", 0, 1)
cut_curve.SetLineColor(2)
cut_curve.SetLineWidth(3)
#cut_curve.Draw("SAME")

cut_curve2 = r.TF1("cut2", str(a1)+"*(1 + exp("+str(b1)+ " *x*x))", 0, 1)
cut_curve2.SetLineColor(2)
cut_curve2.SetLineWidth(3)
#cut_curve2.Draw("SAME")

cut_curve3 = r.TF1("cut_med", str(a2)+"*(1 + exp("+str(b2)+ " *x*x))", 0, 1)
cut_curve3.SetLineColor(2)
cut_curve3.SetLineWidth(3)
cut_curve3.Draw("SAME")

t1 = r.TText(0.2, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()

In [12]:
c = r.TCanvas()
tracks_2.Draw("VR0_av:tan", "k2>=2 ", "COLZ")
c.Draw()

In [13]:
### Numero totale di frammenti
tracks_2.Draw("VR0_av:tan>>Framm_Manuale", 
                   "VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan))", "COLZ")

fm = r.gDirectory.Get("Framm_Manuale")
N_frag = fm.GetEntries()
print("Numero totale di frammenti identificato dal taglio = " + str(N_frag) )

Numero totale di frammenti identificato dal taglio = 32164.0


### Distribuzione VR0_av vs tan solo per le tracce con k1 < 2, k2 < 2 e k3 < 2

In [14]:
c2 = r.TCanvas()
tracks_2.Draw("VR0_av:tan>>Framm_Manuale", 
                   "VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k1 < 2 && k2 <2 && k3 < 2", "COLZ")

fm = r.gDirectory.Get("Framm_Manuale")
fm.SetTitle("VR0_av vs tan(#theta) (Frammenti Z = 1 Alta Energia, k0>=1, k1<2, k2<2, k3<2); tan(#theta); VR0_av")
fm.Draw("COLZ")

r.gPad.Update()
st = fm.FindObject("stats")
st.SetX1NDC(0.50)
st.SetX2NDC(0.70)
st.SetY1NDC(0.70)
st.SetY2NDC(0.90)

c2.Draw()

Queste tracce vengono identificate come particelle con Z = 1 (componente ad alta energia).

In [15]:
k = r.TCanvas()
tracks_2.Draw("tan>>Z_1_high", "VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k1 < 2 && k2<2 && k3<2", "COLZ")

h = r.gDirectory.Get("Z_1_high")
h.SetTitle("Z=1 (High Energy) Fragments Angular Distribution; tan(#theta); Entries")
h.Draw()

Z1_high = h.GetEntries()

k.Draw()

### Stima Incertezza (Z1_high)

In [16]:
k = r.TCanvas()

tracks_2.Draw("tan>>hs", "VR0_av >= " + str(a0) + "*(1 + TMath::Exp(" + str(b0) + " *tan*tan)) && k1 < 2 && k2<2 && k3<2", "COLZ")
h = r.gDirectory.Get("hs")
N1 = h.GetEntries()

tracks_2.Draw("tan>>hs", "VR0_av >= " + str(a1) + "*(1 + TMath::Exp(" + str(b1) + " *tan*tan)) && k1 < 2 && k2<2 && k3<2", "COLZ")
h = r.gDirectory.Get("hs")
N2 = h.GetEntries()

print("Incertezza Sistematica sulle tracce identificate come Z = 1 dal taglio netto: (Max - Min) / 2")
print("N = " + str(Z1_high) + " +/- " + str((N2 - N1)/2))

Incertezza Sistematica sulle tracce identificate come Z = 1 dal taglio netto: (Max - Min) / 2
N = 18555.0 +/- -4579.5


### Scrittura carica su file separato (Z1_high)

In [17]:
file3 = r.TFile("Z_GSI1_I.root", "RECREATE")

Z1_high_E_trk = tracks_2.CopyTree("VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k1 < 2 && k2<2 && k3<2")
Z1_high_E_trk.Write("Z1_hE_trk")
file3.Close()

## Distribuzione VR1_av vs VR0_av

In [18]:
k = r.TCanvas()
tracks_2.Draw("VR1_av:VR0_av>>vr1_vr0(80, 5000,25000, 80, 0, 25000)", "VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k1 >= 2", "COLZ")

h = r.gDirectory.Get("vr1_vr0")
h.SetTitle("VR1_{av} vs VR0_{av} [Fragments, k_{1}>=2); VR0_{av}; VR1_{av}")
h.GetXaxis().SetRangeUser(6000,20500)
h.GetYaxis().SetRangeUser(0, 15500)
h.Draw("COLZ")

k.Draw()

In [19]:
k = r.TCanvas()
tracks_2.Draw("VR1_av:VR0_av>>vr1_vr0(80, 5000,25000, 80, 0, 25000)", "VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k1 >= 2", "COLZ")

h = r.gDirectory.Get("vr1_vr0")
h.SetTitle("VR1_{av} vs VR0_{av} [Fragments, k_{1}>=2]; VR0_{av}; VR1_{av}")
h.GetXaxis().SetRangeUser(6000,20500)
h.GetYaxis().SetRangeUser(0, 15500)
h.Draw("COLZ")

c1 = 4750
c2 = 5000
c0 = (c1+c2)/2
cut_curve2 = r.TF1("cut2", str(c0), 4000, 26000)
cut_curve2.SetLineColor(2)
cut_curve2.SetLineWidth(4)
cut_curve2.Draw("SAME")

t1 = r.TText(15000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

k.Draw()

In [20]:
k = r.TCanvas()
tracks_2.Draw("VR1_av:VR0_av>>vr1_vr0(80, 5000,25000, 80, 0, 25000)", "VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k1 >= 2 && k2<2 && k3<2", "COLZ")

h = r.gDirectory.Get("vr1_vr0")
h.SetTitle("VR1_{av} vs VR0_{av} [Fragments, k_{1}>=2, k_{2}<2, k_{3}<2]; VR0_{av}; VR1_{av}")
h.GetXaxis().SetRangeUser(6000,20500)
h.GetYaxis().SetRangeUser(0, 15500)
h.Draw("COLZ")

c1 = 4750
c2 = 5000
c0 = (c1+c2)/2
cut_curve2 = r.TF1("cut2", str(c0), 4000, 26000)
cut_curve2.SetLineColor(2)
cut_curve2.SetLineWidth(4)
cut_curve2.Draw("SAME")

t1 = r.TText(15000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

k.Draw()

# Clustering VR1:VR0

In [21]:
file_clust = r.TFile("Cluster_GSI1_cuts.root", "RECREATE")

cluster_v1 = tracks_2.CopyTree("VR0_av >= " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) ")
cluster_v2 = tracks_2.CopyTree("k2 >=2 || k1 >=2 || k3 >=2")
cluster_v2.Write("cluster_v1")
file_clust.Close()

In [22]:
#modello = Clustering_GM2("Cluster_GSI1_cuts.root", ['VR1_av', 'VR0_av'], "cluster_v1", 1, 1)

In [23]:
file_clust = r.TFile("Cluster_GSI1_cuts.root", "READ")

c1 = 4500
c2 = 5000
c0 = (c1+c2)/2


cluster_v1 = file_clust.Get("cluster_v1")
cut_curve2 = r.TF1("cut2", str(c0), 4000, 26000)
cut_curve2.SetLineColor(8)
cut_curve2.SetLineWidth(2)

cut_curve3 = r.TF1("cut3", str(c1), 4000, 26000)
cut_curve3.SetLineColor(2)
cut_curve3.SetLineWidth(2)
cut_curve3.Draw("SAME")

cut_curve4 = r.TF1("cut4", str(c2), 4000, 26000)
cut_curve4.SetLineColor(2)
cut_curve4.SetLineWidth(2)
cut_curve4.Draw("SAME")


cluster_v1 = file_clust.Get("cluster_v1")

k = r.TCanvas()
cluster_v1.Draw("VR1_av:VR0_av>>h10", "", "COLZ")

h10 = r.gDirectory.Get("h10")
h10.SetTitle("VR1_av vs VR0_av; VR0_av; VR1_av")
h10.Draw("COLZ")
cut_curve2.Draw("SAME")
cut_curve3.Draw("SAME")
cut_curve4.Draw("SAME")
k.Draw()

In [24]:
file_m = r.TFile("Clustering_man.root", "RECREATE")

man_tuple = r.TNtuple("man_tp", "mn", "n_clust_1")

for track in cluster_v1:
    if (track.VR1_av < c0):
        man_tuple.Fill(1)
    else:
        man_tuple.Fill(0)
man_tuple.Write("man_tp")

373

In [25]:
cluster_v1.AddFriend(man_tuple)

### Componente Z=1 a bassa energia 

In [26]:
k = r.TCanvas()
cluster_v1.Draw("tan>>Z_1_low", "n_clust_1 == 1 && k2 <2 && k3 <2", "COLZ")

h1 = r.gDirectory.Get("Z_1_low")
h1.SetTitle("Z=1 Fragments (Low Energy) Angular Distribution; tan(#theta); Entries")
h1.Draw()

Z1_low = h1.GetEntries()

t1 = r.TText(0.8, 50, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

k.Draw()

### Stima Incertezza

In [27]:
k = r.TCanvas()

cluster_v1.Draw("tan>>h", "VR1_av < " + str(c2) + " && k2<2 && k3<2", "COLZ")
h = r.gDirectory.Get("h")
N1 = h.GetEntries()

cluster_v1.Draw("tan>>h", "VR1_av < " + str(c1)  + " && k2<2 && k3<2", "COLZ")
h = r.gDirectory.Get("h")
N2 = h.GetEntries()

print("Incertezza Sistematica sulle tracce identificate come Z = 1 dal taglio netto: (Max - Min) / 2")
print("N = " + str(Z1_low) + " +/- " + str((N1 - N2)/2))

Incertezza Sistematica sulle tracce identificate come Z = 1 dal taglio netto: (Max - Min) / 2
N = 3805.0 +/- 144.5


In [28]:
k = r.TCanvas()
cluster_v1.Draw("VR1_av:VR0_av>>h(80, 5000,25000, 80, 0, 25000)", " k2<2 && k3<2", "COLZ")

h = r.gDirectory.Get("h")
h.SetTitle("VR1_av vs VR0_av (k1 <= 1); VR0_av; VR1_av")
h.GetXaxis().SetRangeUser(5900,18000)
h.GetYaxis().SetRangeUser(0, 16000)
h.Draw("COLZ")

k.Draw()

### Scrittura Cariche (Z1_low)

In [29]:
file4 = r.TFile("Z_GSI1_I.root", "UPDATE")

Z1_low_E_trk = cluster_v1.CopyTree("n_clust_1 == 1 && k2<2 && k3<2")
Z1_low_E_trk.Write("Z1_lE_trk")

8160

### Componente Z = 2 ad alta energia

In [30]:
k = r.TCanvas()
cluster_v1.Draw("tan>>Z_2_high", "n_clust_1 == 0 && k2<2 && k3<2", "COLZ")

h = r.gDirectory.Get("Z_2_high")
h.SetTitle("Z=2 (High Energy) Angular Distribution; tan(#theta); Entries")
h.Draw()

Z2_high = h.GetEntries()

t1 = r.TText(0.8, 15, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

k.Draw()

In [31]:
k = r.TCanvas()
cluster_v1.Draw("tan>>h(50, 0, 1)", "n_clust_1 == 0 && k2<2 && k3<2 && k0>1", "COLZ")
k.Draw()

### Stima Incertezza

In [32]:
k = r.TCanvas()
cluster_v1.Draw("tan>>h", "VR1_av > " + str(c2) + " && k2<2 && k3<2" , "COLZ")
N1 = h.GetEntries()

cluster_v1.Draw("tan>>h", "VR1_av > " + str(c1) + " && k2<2 && k3<2", "COLZ")
h = r.gDirectory.Get("h")
N2 = h.GetEntries()

print("Incertezza Sistematica sulle tracce identificate come Z = 1 dal taglio netto: (Max - Min) / 2")
print("N = " + str(Z2_high) + " +/- " + str((N1 - N2)/2))

Incertezza Sistematica sulle tracce identificate come Z = 1 dal taglio netto: (Max - Min) / 2
N = 715.0 +/- -83.0


### Scrittura carica 

In [33]:
Z2_high_E_trk = cluster_v1.CopyTree("n_clust_1 == 0 && k2<2 && k3<2")
Z2_high_E_trk.Write("Z2_hE_trk")

7685

In [34]:
## Numero totale di tracce identificate con la Cut Based Analysis
N_cuts = Z2_high + Z1_high + Z1_low 
print("Numero totale di tracce identificate con i tagli finora = " + str(N_cuts))

Numero totale di tracce identificate con i tagli finora = 23075.0


In [35]:
h1.SetFillColor(0)
h1.SetLineColor(4)
h1.SetLineWidth(2)


h.SetFillColor(0)
h.SetLineColor(2)
h.SetLineWidth(2)

hs = r.THStack("hs", "hs")
hs.Add(h1)
hs.Add(h)

hs.SetTitle("Angular Distribution [Fragments, k_{2}<2 & k_{3}<2]; tan(#theta);Entries")
c = r.TCanvas()
hs.Draw("nostack")
c.Draw()

### Check: Quante Tracce Mancano

In [36]:
cluster_v1.Draw("VR0_av>>h12", "k1>=1 && k2>=2 && k3>=1 || k1>=1 && k2>=1 && k3>=2")
h12 = r.gDirectory.Get("h12")
print(h12.GetEntries())
cluster_v1.Draw("VR0_av>>h12", "k1>=1 && k2>=2 && k3<1")
h12 = r.gDirectory.Get("h12")
print(h12.GetEntries())
cluster_v1.Draw("VR0_av>>h12", "k1>=1 && k2<1 && k3>=2")
h12 = r.gDirectory.Get("h12")
print(h12.GetEntries())
cluster_v1.Draw("VR0_av>>h12", "k1<1 && k2>=1 && k3>=1")
h12 = r.gDirectory.Get("h12")
print(h12.GetEntries())

9618.0
242.0
113.0
34.0


In [37]:
N_frag

32164.0

In [38]:
file4.Close()

In [44]:
c = r.TCanvas()

cluster_v1.Draw("VR2_av:VR1_av>>vr2_vr1", "k2>=2 && k1>=2 && k0>=7", "COLZ")

v21 = r.gDirectory.Get("vr2_vr1")
v21.SetTitle("VR2_{av} vs VR1_{av} [k_{1}>=2 && k_{2}>=2 && k_{0}>=7];VR1_{av};VR2_{av}")
v21.Draw("COLZ")

t1 = r.TText(15000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()

In [40]:
c = r.TCanvas()

cluster_v1.Draw("VR2_av:VR1_av>>vr2_vr1", "k2>=2 && k1>=2 && k0>=7", "COLZ")

v21 = r.gDirectory.Get("vr2_vr1")
v21.SetTitle("VR2_{av} vs VR1_{av} [k_{1}>=2 & k_{2}>=2 & k_{0}>=7];VR1_{av};VR2_{av}")
v21.Draw("COLZ")

t1 = r.TText(15000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()

In [41]:
c = r.TCanvas()

cluster_v1.Draw("k1", "k2>=2 && k1>=2 && VR2_av>9800", "COLZ")
c.Draw()

In [42]:
c = r.TCanvas()

cluster_v1.Draw("VR3_av:VR2_av>>vr3_vr2", "k2>=2 && k3>=2", "COLZ")

v21 = r.gDirectory.Get("vr3_vr2")
v21.SetTitle("VR3_{av} vs VR2_{av} [k_{2}>=2 && k_{3}>=2];VR2_{av};VR3_{av}")
v21.Draw("COLZ")

t1 = r.TText(10000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()

# Classificazione k2>=2 && k3>=2 (VR123)

In [43]:
r.Math.MinimizerOptions.SetDefaultMinimizer("Genetic")
r.Math.MinimizerOptions.SetDefaultTolerance(1e-10)

In [44]:
c = r.TCanvas()

cluster_v1.Draw("VR2_av:VR1_av", "k2>=2", "COLZ")
c.Draw()

In [45]:
file_pca = r.TFile("GSI1_PCA.root", "RECREATE")

campione_pca = cluster_v1.CopyTree("k1 >=1 && k2>=2 && k3>=1 || k1>=1 && k2>=1 && k3>=2")

campione_pca.Write("pca")

file_pca.Close()

In [46]:
file_clust.Close()
file_m.Close()

In [47]:
principal = r.TPrincipal(3, "ND")

file_pca = r.TFile("GSI1_PCA.root", "READ")
info_pca = file_pca.Get("pca")

for track in info_pca:
    vr0, vr1, vr2 = track.VR1_av, track.VR2_av, track.VR3_av
    vrs = np.zeros(3)
    vrs[0] = vr0
    vrs[1] = vr1
    vrs[2] = vr2
    principal.AddRow(vrs)
    
principal.MakePrincipals()
principal.MakeCode()

Writing on file "pca.C" ... done


In [48]:
r.gSystem.Load("pca_C.so")
vr123s = []
for track in info_pca:
    vr2, vr3, vr0 = track.VR1_av, track.VR2_av, track.VR3_av
    vrs = np.zeros(3)
    vrs[0] = vr2
    vrs[1] = vr3
    vrs[2] = vr0
    princ = np.zeros(3)
    principal.X2P(vrs, princ)
    vr123s.append(princ)

vr123 = []
for i in vr123s:
    vr123.append(i[0])
    
file_pca.Close()

In [49]:
file_pca_2 = r.TFile("pca_GSI1.root", "RECREATE")

pca_1 = r.TNtuple("pca_1", "", "VR123")

for i in range(len(vr123)):
    pca_1.Fill(vr123[i])

In [50]:
kn = r.TCanvas()
pca_1.Draw("VR123>>123(60, -4., 4.5.)")

hi = r.gDirectory.Get("123")
hi.SetTitle("VP_{123} [k_{1}, k_{3}>=1 & k_{2}>=2 or k_{1}, k_{2}>=1 & k_{3}>=2]; VP_{123}; Entries")
hi.Draw("PE")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
legend.AddEntry("Cut", "Cut: k1 >=1 && k2>=2 && k3>=1 || k1>=1 && k2>=1 && k3>=2", "")
#legend.Draw("SAME")

t1 = r.TText(2, 200, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

kn.Draw()

In [51]:
pca_1.Write("pca")

366

### Fit Senza Soglia

In [52]:
pca_1.Draw("VR123>>v123s(60, -3.5, 5.5)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VR123; VR123; Conteggi")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(120, -1.5, .5, 60, 0.9, 0.7, 40, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 80, 600)
fit_func.SetParLimits(3, 20, 250)
fit_func.SetParLimits(6, 20, 150)

#punto medio
fit_func.SetParLimits(1, -2.3, -1.)
fit_func.SetParLimits(4, 0., 1.5)
fit_func.SetParLimits(7, 2., 5.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.)
fit_func.SetParLimits(5, 0., 2)
fit_func.SetParLimits(8, 0., 2.)


histos.Fit("fit_func", "S", "", -4., 7.)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(6)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()))
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()) )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)))
legend.Draw("SAME")

c.Draw()


****************************************
Minimizer is Genetic / Migrad
Chi2                      =      159.266
NDf                       =           41
Edm                       =            0
NCalls                    =        30300
p0                        =      468.536 	 (limited)
p1                        =     -1.35761 	 (limited)
p2                        =     0.545269 	 (limited)
p3                        =       211.16 	 (limited)
p4                        =     0.663248 	 (limited)
p5                        =      1.19042 	 (limited)
p6                        =      149.992 	 (limited)
p7                        =      2.83455 	 (limited)
p8                        =     0.387079 	 (limited)


### Soglia 1, Binning 1

In [53]:
r.Math.MinimizerOptions.SetDefaultMinimizer("Genetic")
r.Math.MinimizerOptions.SetDefaultTolerance(1e-10)
r.Math.MinimizerOptions.SetDefaultMaxIterations(1000)

In [54]:
pca_1.Draw("VR123>>v123s(70, -3.5, 5.5)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VR123; VR123; Conteggi")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4., 6.)
fit_func.SetParameters(350, -1.5, .5, 110, .5, 1., 110, 3.5, 1.)

#ampiezze
fit_func.SetParLimits(0, 200, 400)
fit_func.SetParLimits(3, 50, 250)
fit_func.SetParLimits(6, 20, 150)

#punto medio
fit_func.SetParLimits(1, -2.5, -1.1)
fit_func.SetParLimits(4, -.5, 1.8)
fit_func.SetParLimits(7, 3.5, 5.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.)
fit_func.SetParLimits(5, 0., 2)
fit_func.SetParLimits(8, 0., 1.)

tr = -2.

histos.Fit("fit_func", "S", "", tr, 6.)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")
c.Update()

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -4, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", tr, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(6)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", tr, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

ymax = r.gPad.GetUymax()

line = r.TLine(tr, 0, tr, ymax)
line.SetLineColor(12)
line.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()))
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()) )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)))
legend.Draw("SAME")

c.Draw()


****************************************
Minimizer is Genetic / Migrad
Chi2                      =        490.2
NDf                       =           40
Edm                       =            0
NCalls                    =       300300
p0                        =      376.406 	 (limited)
p1                        =     -1.36119 	 (limited)
p2                        =      1.09853 	 (limited)
p3                        =      161.003 	 (limited)
p4                        =          1.8 	 (limited)
p5                        =     0.887447 	 (limited)
p6                        =      27.2033 	 (limited)
p7                        =      3.52538 	 (limited)
p8                        =       0.0246 	 (limited)


### Tentativo con Python

In [ ]:
from lmfit import Minimizer, Parameters, report_fit
from lmfit.lineshapes import gaussian, lorentzian


def residual(pars, x, data):
    model = (gaussian(x, pars['amp_g1'], pars['cen_g1'], pars['wid_g1']) +
             gaussian(x, pars['amp_g2'], pars['cen_g2'], pars['wid_g2']) +
            gaussian(x, pars['amp_g3'], pars['cen_g3'], pars['wid_g3'])
            ) 
    return model - data

In [ ]:
vr123_tr = []

for el in vr123:
    if (el > tr):
        vr123_tr.append(el)

In [ ]:
counts, bin_edges = np.histogram(vr123_tr, bins=80)
bin_widths = np.diff(bin_edges)
x = bin_edges[:-1] + (bin_widths / 2)
y = counts

In [ ]:
pfit = Parameters()

pfit.add(name='amp_g3', value=80, vary = True, min=30, max=160)
pfit.add(name='peak_split', value=50, min=0, max=200, vary=True)
pfit.add(name='peak_split2', value=50, min=50, max=200, vary=True)
pfit.add(name='amp_g2', expr='amp_g3 + peak_split', min=30, max=180)
pfit.add(name='amp_g1', expr='amp_g2 + peak_split2', min=100, max=500)

pfit.add(name='cen_g1', value=-1.5, vary=True, min=-2., max=-1.)
pfit.add(name='cen_g2', value=.5, vary=True, min=0., max=1.5)
pfit.add(name='cen_g3', value=3.5, vary=True, min=2., max=5.)

pfit.add(name='wid_g1', value=.5, vary=True, min=0.1, max=2.)
pfit.add(name='wid_g2', value=1., vary=True, min=0.1, max=1.)
pfit.add(name='wid_g3', value=.5, vary=True, min=0.1, max=1.)


mini = Minimizer(residual, pfit, fcn_args=(x, y))
out = mini.leastsq()
best_fit = y + out.residual

In [ ]:
plt.plot(x, y, 'o')
plt.plot(x, best_fit, '--', label='best fit')
plt.legend()

In [ ]:
import lmfit
o2 = lmfit.minimize(residual, pfit, args=(x, y), method='differential_evolution')
print("\n\n# Fit using differential_evolution:")
lmfit.report_fit(o2)

plt.plot(x, y, 'o')
plt.plot(x, y+out.residual, '--', label='best fit')
plt.legend()
plt.show()

In [ ]:
t1 = time.time()

N = 100

cl1, cl2, cl3 = [], [], []
x = np.linspace(min(vr123), max(vr123), 1000)

for i in range(N):
    histos.Fit("fit_func", "S", "", tr, 7.)

    params = fit_func.GetParameters()

    if (fit_func.GetProb() > 0.0001):

        plt.plot(x, g_func(x, params[0], params[1], params[2]), color = 'b')
        plt.plot(x, g_func(x, params[3], params[4], params[5]), color = 'yellow')
        plt.plot(x, g_func(x, params[6], params[7], params[8]), color = 'turquoise')
        plt.plot(x, g_func(x, params[0], params[1], params[2]) + g_func(x, params[3], params[4], params[5]) + g_func(x, params[6], params[7], params[8]), color = 'red' )
        
        cn1, cn2, cn3 = [], [], []
        
        comp1.SetParameters(params[0], params[1], params[2])
        comp2.SetParameters(params[3], params[4], params[5])
        comp3.SetParameters(params[6], params[7], params[8])

     
        for value in vr123:
            if (value < tr):
                cn1.append(value)
            else:
                random_number = np.random.uniform(0,1)
                p1, p2, p3 = comp1.Eval(value)/fit_func.Eval(value), comp2.Eval(value)/fit_func.Eval(value), comp3.Eval(value)/fit_func.Eval(value)
                p_s = [p1, p2, p3]
                o_ps = sorted(p_s)
                id_1, id_2, id_3 = o_ps.index(p1), o_ps.index(p2), o_ps.index(p3)
    
                if (random_number <= o_ps[0]):
                    if (id_1 == 0 and id_2!=0 and id_3!=0):
                        cn1.append(value)
            
                    elif(id_1 != 0 and id_2==0 and id_3 !=0):
                        cn2.append(value)
             
                    elif(id_1 != 0 and id_2!=0 and id_3 ==0):
                        cn3.append(value)
         
                elif(random_number > o_ps[0] and random_number <= o_ps[1]+o_ps[0]):
        
                    if (id_1 == 1 and id_2!=1 and id_3!=1):
                        cn1.append(value)
            
                    elif(id_1 != 1 and id_2==1 and id_3 !=1):
                        cn2.append(value)
            
                    elif(id_1 != 1 and id_2!=1 and id_3 ==1):
                        cn3.append(value)
            
                elif(random_number > o_ps[1]+o_ps[0]):
                    if (id_1 == 2 and id_2!=2 and id_3!=2):
                        cn1.append(value)
            
                    elif(id_1 != 2 and id_2==2 and id_3 !=2):
                        cn2.append(value)
            
                    elif(id_1 != 2 and id_2!=2 and id_3 ==2):
                        cn3.append(value)   

        cl1.append(len(cn1))
        cl2.append(len(cn2))
        cl3.append(len(cn3))
    if (i%100 == 0):
        print(i)

plt.xlabel("VR123")
plt.ylabel("Conteggi")
plt.grid(True)
plt.title("Fluttuazione nel fit")
plt.show()

print(time.time()-t1)

In [ ]:
from statistics import mean
print("La media dei conteggi della popolazione Z = 2 è " + str(round(mean(cl1), 1)) + " con deviazione standard: " + str(np.std(cl1)))
print("La media dei conteggi della popolazione Z = 3 è " + str(round(mean(cl2), 1)) + " con deviazione standard: " + str(np.std(cl2)))
print("La media dei conteggi della popolazione Z > 3 è " + str(round(mean(cl3), 1)) + " con deviazione standard: " + str(np.std(cl3)))

In [ ]:
Z2_means, Z3_means, Z4_means = [], [], []
Z2_stds, Z3_stds, Z4_stds = [], [], [] 

Z2_means.append(mean(cl1))
Z3_means.append(mean(cl2))
Z4_means.append(mean(cl3))

Z2_stds.append(np.std(cl1))
Z3_stds.append(np.std(cl2))
Z4_stds.append(np.std(cl3))

### Soglia 1, Binning 2

In [ ]:
r.Math.MinimizerOptions.SetDefaultMinimizer("Genetic")
r.Math.MinimizerOptions.SetDefaultTolerance(1e-10)
r.Math.MinimizerOptions.SetDefaultMaxIterations(200)

In [ ]:
pca_1.Draw("VR123>>v123s(100, -3.5., 5.)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VR123; VR123; Conteggi")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(501, -1.5, .5, 110, 0.9, 0.7, 110, 3.5, 0.5)

#ampiezze
fit_func.SetParLimits(0, 500, 650)
fit_func.SetParLimits(3, 100, 200)
fit_func.SetParLimits(6, 100, 200)

#punto medio
fit_func.SetParLimits(1, -2.5, -1.5)
fit_func.SetParLimits(4, -1, 2.)
fit_func.SetParLimits(7, 2., 4.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.5)
fit_func.SetParLimits(5, 0.2, 2)
fit_func.SetParLimits(8, 0.3, 2.5)


histos.Fit("fit_func", "", "", tr, 7.)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(6)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()))
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()) )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)))
legend.Draw("SAME")

c.Draw()

In [ ]:
t1 = time.time()

N = 100

cl1, cl2, cl3 = [], [], []
x = np.linspace(min(vr123), max(vr123), 1000)

for i in range(N):
    histos.Fit("fit_func", "S", "", tr, 7.)

    params = fit_func.GetParameters()

    if (fit_func.GetProb() > 0.0001):

        plt.plot(x, g_func(x, params[0], params[1], params[2]), color = 'b')
        plt.plot(x, g_func(x, params[3], params[4], params[5]), color = 'yellow')
        plt.plot(x, g_func(x, params[6], params[7], params[8]), color = 'turquoise')
        plt.plot(x, g_func(x, params[0], params[1], params[2]) + g_func(x, params[3], params[4], params[5]) + g_func(x, params[6], params[7], params[8]), color = 'red' )
        
        cn1, cn2, cn3 = [], [], []
        
        comp1.SetParameters(params[0], params[1], params[2])
        comp2.SetParameters(params[3], params[4], params[5])
        comp3.SetParameters(params[6], params[7], params[8])

     
        for value in vr123:
            if (value < tr):
                cn1.append(value)
            else:
                random_number = np.random.uniform(0,1)
                p1, p2, p3 = comp1.Eval(value)/fit_func.Eval(value), comp2.Eval(value)/fit_func.Eval(value), comp3.Eval(value)/fit_func.Eval(value)
                p_s = [p1, p2, p3]
                o_ps = sorted(p_s)
                id_1, id_2, id_3 = o_ps.index(p1), o_ps.index(p2), o_ps.index(p3)
    
                if (random_number <= o_ps[0]):
                    if (id_1 == 0 and id_2!=0 and id_3!=0):
                        cn1.append(value)
            
                    elif(id_1 != 0 and id_2==0 and id_3 !=0):
                        cn2.append(value)
             
                    elif(id_1 != 0 and id_2!=0 and id_3 ==0):
                        cn3.append(value)
         
                elif(random_number > o_ps[0] and random_number <= o_ps[1]+o_ps[0]):
        
                    if (id_1 == 1 and id_2!=1 and id_3!=1):
                        cn1.append(value)
            
                    elif(id_1 != 1 and id_2==1 and id_3 !=1):
                        cn2.append(value)
            
                    elif(id_1 != 1 and id_2!=1 and id_3 ==1):
                        cn3.append(value)
            
                elif(random_number > o_ps[1]+o_ps[0]):
                    if (id_1 == 2 and id_2!=2 and id_3!=2):
                        cn1.append(value)
            
                    elif(id_1 != 2 and id_2==2 and id_3 !=2):
                        cn2.append(value)
            
                    elif(id_1 != 2 and id_2!=2 and id_3 ==2):
                        cn3.append(value)   

        cl1.append(len(cn1))
        cl2.append(len(cn2))
        cl3.append(len(cn3))
    if (i%100 == 0):
        print(i)

plt.xlabel("VR123")
plt.ylabel("Conteggi")
plt.grid(True)
plt.title("Fluttuazione nel fit")
plt.show()

print(time.time()-t1)

In [ ]:
print("La media dei conteggi della popolazione Z = 2 è " + str(round(mean(cl1), 1)) + " con deviazione standard: " + str(np.std(cl1)))
print("La media dei conteggi della popolazione Z = 3 è " + str(round(mean(cl2), 1)) + " con deviazione standard: " + str(np.std(cl2)))
print("La media dei conteggi della popolazione Z > 3 è " + str(round(mean(cl3), 1)) + " con deviazione standard: " + str(np.std(cl3)))

In [ ]:
Z2_means.append(mean(cl1))
Z3_means.append(mean(cl2))
Z4_means.append(mean(cl3))

Z2_stds.append(np.std(cl1))
Z3_stds.append(np.std(cl2))
Z4_stds.append(np.std(cl3))

### Soglia 2, Binning 1

In [ ]:
r.Math.MinimizerOptions.SetDefaultMinimizer("Genetic")
r.Math.MinimizerOptions.SetDefaultTolerance(1e-10)
r.Math.MinimizerOptions.SetDefaultMaxIterations(200)

In [ ]:
pca_1.Draw("VR123>>v123s(90, -3.5., 5.)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VR123; VR123; Conteggi")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(501, -1.5, .5, 110, 0.9, 0.7, 110, 3.5, 0.5)

#ampiezze
fit_func.SetParLimits(0, 350, 650)
fit_func.SetParLimits(3, 100, 200)
fit_func.SetParLimits(6, 100, 200)

#punto medio
fit_func.SetParLimits(1, -2.5, -1.)
fit_func.SetParLimits(4, -1, 2.)
fit_func.SetParLimits(7, 2., 4.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.5)
fit_func.SetParLimits(5, 0.2, 2)
fit_func.SetParLimits(8, 0.3, 2.5)

tr = -1.9
histos.Fit("fit_func", "S", "", tr, 7)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(6)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()))
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()) )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)))
legend.Draw("SAME")

c.Draw()

In [ ]:
t1 = time.time()

N = 100

cl1, cl2, cl3 = [], [], []
x = np.linspace(min(vr123), max(vr123), 1000)

for i in range(N):
    histos.Fit("fit_func", "S", "", tr, 7.)

    params = fit_func.GetParameters()

    if (fit_func.GetProb() > 0.001):

        plt.plot(x, g_func(x, params[0], params[1], params[2]), color = 'b')
        plt.plot(x, g_func(x, params[3], params[4], params[5]), color = 'yellow')
        plt.plot(x, g_func(x, params[6], params[7], params[8]), color = 'turquoise')
        plt.plot(x, g_func(x, params[0], params[1], params[2]) + g_func(x, params[3], params[4], params[5]) + g_func(x, params[6], params[7], params[8]), color = 'red' )
        
        cn1, cn2, cn3 = [], [], []
        
        comp1.SetParameters(params[0], params[1], params[2])
        comp2.SetParameters(params[3], params[4], params[5])
        comp3.SetParameters(params[6], params[7], params[8])

     
        for value in vr123:
            if (value < tr):
                cn1.append(value)
            else:
                random_number = np.random.uniform(0,1)
                p1, p2, p3 = comp1.Eval(value)/fit_func.Eval(value), comp2.Eval(value)/fit_func.Eval(value), comp3.Eval(value)/fit_func.Eval(value)
                p_s = [p1, p2, p3]
                o_ps = sorted(p_s)
                id_1, id_2, id_3 = o_ps.index(p1), o_ps.index(p2), o_ps.index(p3)
    
                if (random_number <= o_ps[0]):
                    if (id_1 == 0 and id_2!=0 and id_3!=0):
                        cn1.append(value)
            
                    elif(id_1 != 0 and id_2==0 and id_3 !=0):
                        cn2.append(value)
             
                    elif(id_1 != 0 and id_2!=0 and id_3 ==0):
                        cn3.append(value)
         
                elif(random_number > o_ps[0] and random_number <= o_ps[1]+o_ps[0]):
        
                    if (id_1 == 1 and id_2!=1 and id_3!=1):
                        cn1.append(value)
            
                    elif(id_1 != 1 and id_2==1 and id_3 !=1):
                        cn2.append(value)
            
                    elif(id_1 != 1 and id_2!=1 and id_3 ==1):
                        cn3.append(value)
            
                elif(random_number > o_ps[1]+o_ps[0]):
                    if (id_1 == 2 and id_2!=2 and id_3!=2):
                        cn1.append(value)
            
                    elif(id_1 != 2 and id_2==2 and id_3 !=2):
                        cn2.append(value)
            
                    elif(id_1 != 2 and id_2!=2 and id_3 ==2):
                        cn3.append(value)   

        cl1.append(len(cn1))
        cl2.append(len(cn2))
        cl3.append(len(cn3))
    if (i%100 == 0):
        print(i)

plt.xlabel("VR123")
plt.ylabel("Conteggi")
plt.grid(True)
plt.title("Fluttuazione nel fit")
plt.show()

print(time.time()-t1)

In [ ]:
print("La media dei conteggi della popolazione Z = 2 è " + str(round(mean(cl1), 1)) + " con deviazione standard: " + str(np.std(cl1)))
print("La media dei conteggi della popolazione Z = 3 è " + str(round(mean(cl2), 1)) + " con deviazione standard: " + str(np.std(cl2)))
print("La media dei conteggi della popolazione Z > 3 è " + str(round(mean(cl3), 1)) + " con deviazione standard: " + str(np.std(cl3)))

In [ ]:
Z2_means.append(mean(cl1))
Z3_means.append(mean(cl2))
Z4_means.append(mean(cl3))

Z2_stds.append(np.std(cl1))
Z3_stds.append(np.std(cl2))
Z4_stds.append(np.std(cl3))

### Soglia 2, Binning 2

In [ ]:
r.Math.MinimizerOptions.SetDefaultMinimizer("Genetic")
r.Math.MinimizerOptions.SetDefaultTolerance(1e-10)
r.Math.MinimizerOptions.SetDefaultMaxIterations(200)

In [ ]:
pca_1.Draw("VR123>>v123s(60, -4., 7.)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VR123; VR123; Conteggi")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(400, -1.5, .5, 100, 1., 0.5, 30, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 250, 500)
fit_func.SetParLimits(3, 60, 150)
fit_func.SetParLimits(6, 15, 40)

#punto medio
fit_func.SetParLimits(1, -2.5, -1.4)
fit_func.SetParLimits(4, -0.5, 2.)
fit_func.SetParLimits(7, 3.5, 4.5)

#deviazione_st
fit_func.SetParLimits(2, 0., 1.5)
fit_func.SetParLimits(5, 0., 2)
fit_func.SetParLimits(8, 0., 1.5)


histos.Fit("fit_func", "S", "", tr, 7)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(6)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()))
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()) )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)))
legend.Draw("SAME")

c.Draw()

In [ ]:
t1 = time.time()

N = 200

cl1, cl2, cl3 = [], [], []
x = np.linspace(min(vr123), max(vr123), 1000)

for i in range(N):
    histos.Fit("fit_func", "S", "", tr, 7.)

    params = fit_func.GetParameters()

    if (fit_func.GetProb() > 0.001):

        plt.plot(x, g_func(x, params[0], params[1], params[2]), color = 'b')
        plt.plot(x, g_func(x, params[3], params[4], params[5]), color = 'yellow')
        plt.plot(x, g_func(x, params[6], params[7], params[8]), color = 'turquoise')
        plt.plot(x, g_func(x, params[0], params[1], params[2]) + g_func(x, params[3], params[4], params[5]) + g_func(x, params[6], params[7], params[8]), color = 'red' )
        
        cn1, cn2, cn3 = [], [], []
        
        comp1.SetParameters(params[0], params[1], params[2])
        comp2.SetParameters(params[3], params[4], params[5])
        comp3.SetParameters(params[6], params[7], params[8])

     
        for value in vr123:
            if (value < tr):
                cn1.append(value)
            else:
                random_number = np.random.uniform(0,1)
                p1, p2, p3 = comp1.Eval(value)/fit_func.Eval(value), comp2.Eval(value)/fit_func.Eval(value), comp3.Eval(value)/fit_func.Eval(value)
                p_s = [p1, p2, p3]
                o_ps = sorted(p_s)
                id_1, id_2, id_3 = o_ps.index(p1), o_ps.index(p2), o_ps.index(p3)
    
                if (random_number <= o_ps[0]):
                    if (id_1 == 0 and id_2!=0 and id_3!=0):
                        cn1.append(value)
            
                    elif(id_1 != 0 and id_2==0 and id_3 !=0):
                        cn2.append(value)
             
                    elif(id_1 != 0 and id_2!=0 and id_3 ==0):
                        cn3.append(value)
         
                elif(random_number > o_ps[0] and random_number <= o_ps[1]+o_ps[0]):
        
                    if (id_1 == 1 and id_2!=1 and id_3!=1):
                        cn1.append(value)
            
                    elif(id_1 != 1 and id_2==1 and id_3 !=1):
                        cn2.append(value)
            
                    elif(id_1 != 1 and id_2!=1 and id_3 ==1):
                        cn3.append(value)
            
                elif(random_number > o_ps[1]+o_ps[0]):
                    if (id_1 == 2 and id_2!=2 and id_3!=2):
                        cn1.append(value)
            
                    elif(id_1 != 2 and id_2==2 and id_3 !=2):
                        cn2.append(value)
            
                    elif(id_1 != 2 and id_2!=2 and id_3 ==2):
                        cn3.append(value)   

        cl1.append(len(cn1))
        cl2.append(len(cn2))
        cl3.append(len(cn3))
    if (i%100 == 0):
        print(i)

plt.xlabel("VR123")
plt.ylabel("Conteggi")
plt.grid(True)
plt.title("Fluttuazione nel fit")
plt.show()

print(time.time()-t1)

In [ ]:
print("La media dei conteggi della popolazione Z = 2 è " + str(round(mean(cl1), 1)) + " con deviazione standard: " + str(np.std(cl1)))
print("La media dei conteggi della popolazione Z = 3 è " + str(round(mean(cl2), 1)) + " con deviazione standard: " + str(np.std(cl2)))
print("La media dei conteggi della popolazione Z > 3 è " + str(round(mean(cl3), 1)) + " con deviazione standard: " + str(np.std(cl3)))

In [ ]:
Z2_means.append(mean(cl1))
Z3_means.append(mean(cl2))
Z4_means.append(mean(cl3))

Z2_stds.append(np.std(cl1))
Z3_stds.append(np.std(cl2))
Z4_stds.append(np.std(cl3))

### Soglia 3, Binning 1

In [ ]:
r.Math.MinimizerOptions.SetDefaultMinimizer("Genetic")
r.Math.MinimizerOptions.SetDefaultTolerance(1e-10)
r.Math.MinimizerOptions.SetDefaultMaxIterations(200)

In [ ]:
pca_1.Draw("VR123>>v123s(70, -4., 7.)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VR123; VR123; Conteggi")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(300, -1.5, .5, 100, 1., 0.5, 30, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 280, 500)
fit_func.SetParLimits(3, 60, 200)
fit_func.SetParLimits(6, 15, 40)

#punto medio
fit_func.SetParLimits(1, -3.5, -1.2)
fit_func.SetParLimits(4, -0.5, 2.)
fit_func.SetParLimits(7, 3.5, 4.5)

#deviazione_st
fit_func.SetParLimits(2, 0., 1.)
fit_func.SetParLimits(5, 0., 2)
fit_func.SetParLimits(8, 0., 1.5)

tr = -1.96

histos.Fit("fit_func", "S", "", tr, 7.)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(6)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()))
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()) )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)))
legend.Draw("SAME")

c.Draw()

### Media ed Errori

In [ ]:
def weighted_av(values, errors):
    weights = []
    s, W = 0, 0
    for er in errors:
        weights.append(1 / (er*er))
    for i in range(len(values)):
        s += values[i]*weights[i]
        W += weights[i]
    return s/W, 1 / np.sqrt(W)

In [ ]:
print("############  Media Pesata, Deviazione Standard, Errore Sistematico (Max-Min)/2 ##########")
print(weighted_av(Z2_means, Z2_stds))
print(max(Z2_means) - min(Z2_means))

### Fit Scelto

In [55]:
pca_1.Draw("VR123>>v123s(60, -4., 4.5.)", "", "COLZ")

r.gStyle.SetOptStat(0)
histos = r.gDirectory.Get("v123s")
histos.SetTitle("VP_{123} [GSI1]; VP_{123}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(501, -1.5, .5, 110, 0.9, 0.7, 110, 3.5, 0.5)

#ampiezze
fit_func.SetParLimits(0, 400, 520)
fit_func.SetParLimits(3, 100, 200)
fit_func.SetParLimits(6, 100, 200)

#punto medio
fit_func.SetParLimits(1, -1.6, -1.3)
fit_func.SetParLimits(4, 0., 1.2)
fit_func.SetParLimits(7, 2., 4.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.)
fit_func.SetParLimits(5, 0.2, 2)
fit_func.SetParLimits(8, 0.3, 2.5)


tr = -1.8
histos.Fit("fit_func", "", "", tr, 4.5)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", tr, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

comp4 = r.TF1("comp4", "[0]*TMath::Gaus(x, [1], [2])", -4., tr)
comp4.SetParameters(params[0], params[1], params[2])
comp4.SetLineColor(4)
comp4.SetLineStyle(2)
comp4.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Overall Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "")
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)), "")
legend.Draw("SAME")

line = r.TLine(tr, 0, tr, 563)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

t1 = r.TText(10000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()


****************************************
Minimizer is Genetic / Migrad
Chi2                      =      70.4469
NDf                       =           34
Edm                       =            0
NCalls                    =       300300
p0                        =      472.009 	 (limited)
p1                        =         -1.6 	 (limited)
p2                        =      0.98529 	 (limited)
p3                        =       171.45 	 (limited)
p4                        =          1.2 	 (limited)
p5                        =     0.946636 	 (limited)
p6                        =      155.726 	 (limited)
p7                        =       2.8957 	 (limited)
p8                        =     0.358407 	 (limited)


In [56]:
cn1, cn2, cn3 = [], [], []

cn1_t, cn2_t, cn3_t = r.TNtuple("cn1_t22", "", "cl10"), r.TNtuple("cn2_t22", "", "cl20"), r.TNtuple("cn3_t22", "", "cl30")

for value in vr123:
    if (value < tr):
        cn1.append(value)
        cn1_t.Fill(1)
        cn2_t.Fill(0)
        cn3_t.Fill(0)
    else:
        random_number = np.random.uniform(0,1)
        p1, p2, p3 = comp1.Eval(value)/fit_func.Eval(value), comp2.Eval(value)/fit_func.Eval(value), comp3.Eval(value)/fit_func.Eval(value)
        p_s = [p1, p2, p3]
        o_ps = sorted(p_s)
        id_1, id_2, id_3 = o_ps.index(p1), o_ps.index(p2), o_ps.index(p3)
    
        if (random_number <= o_ps[0]):
            if (id_1 == 0 and id_2!=0 and id_3!=0):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 0 and id_2==0 and id_3 !=0):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 0 and id_2!=0 and id_3 ==0):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
            
        elif(random_number > o_ps[0] and random_number <= o_ps[1]+o_ps[0]):
        
            if (id_1 == 1 and id_2!=1 and id_3!=1):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 1 and id_2==1 and id_3 !=1):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 1 and id_2!=1 and id_3 ==1):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
            
        elif(random_number > o_ps[1]+o_ps[0]):
            if (id_1 == 2 and id_2!=2 and id_3!=2):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 2 and id_2==2 and id_3 !=2):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 2 and id_2!=2 and id_3 ==2):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
    

In [57]:
cn1_t.Write("cn1tr")
cn2_t.Write("cn2tr")
cn3_t.Write("cn3tr")
file_pca_2.Close()

In [58]:
file_pca2 = r.TFile("pca_GSI1.root", "READ")

cn1_tr = file_pca2.Get("cn1tr")
cn2_tr = file_pca2.Get("cn2tr")
cn3_tr = file_pca2.Get("cn3tr")

file_pca = r.TFile("GSI1_PCA.root", "READ")

pca = file_pca.Get("pca")

pca.AddFriend(cn1_tr)
pca.AddFriend(cn2_tr)
pca.AddFriend(cn3_tr)

file4 = r.TFile("Z_GSI1_II.root", "RECREATE")

Z2_2_trk = pca.CopyTree("cl10==1 && cl20 == 0 && cl30 == 0")

Z2_2_trk.Write("Z2_2_trk")

z2_2, z3_2, z4_2 = [], [], []

z2_2_tan, z3_2_tan, z4_2_tan = [], [], []

for track in Z2_2_trk:
    z2_2.append(track.trid)
    z2_2_tan.append(track.tan)

In [59]:
file4.Close()

In [60]:
file5 = r.TFile("Z_GSI1_II_2.root", "RECREATE")

Z3_2_trk = pca.CopyTree("cl10==0 && cl20 == 1 && cl30 == 0")
Z3_2_trk.Write("Z3_2_trk")

for track in Z3_2_trk:
    z3_2.append(track.trid)
    z3_2_tan.append(track.tan)
file5.Close()

In [61]:
file6 = r.TFile("Z_GSI1_II_3.root", "RECREATE")

Z4_2_trk = pca.CopyTree("cl10==0 && cl20 == 0 && cl30 == 1")

Z4_2_trk.Write("Z4_2_trk")

for track in Z4_2_trk:
    z4_2.append(track.trid)
    z4_2_tan.append(track.tan)
file6.Close()

In [62]:
check = z2_2 + z3_2 + z4_2

s = 0

for e in check:
    if (check.count(e) != 1):
        s+=1

print(s)
print(len(z2_2_tan))
print(len(z3_2_tan))
print(len(z4_2_tan))
print(len(check))

0
5729
2873
1016
9618


In [63]:
file_pca.Close()

In [64]:
file4.Close()

In [65]:
h4 = r.TH1F("h4", "", 100, -4, 6)
h5 = r.TH1F("h5", "", 100, -4, 6)
h6 = r.TH1F("h6", "", 100, -4, 6)

for i in cn1:
    h4.Fill(i)
for i in cn2:
    h5.Fill(i)
for i in cn3:
    h6.Fill(i)
        
hs = r.THStack("hs", "")
hs.Add(h4)
hs.Add(h6)
hs.Add(h5)

sp_canvas = r.TCanvas()
hs.SetTitle("VR123 (3 Gaussian Fit); VR123; Conteggi")
hs.Draw("pfc nostack")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
legend.AddEntry(h4, "Entries: "+ str(h4.GetEntries()), "f")
legend.AddEntry(h5, "Entries: "+ str(h5.GetEntries()), "f")
legend.AddEntry(h6, "Entries: "+ str(h6.GetEntries()), "f")
legend.Draw("SAME")
sp_canvas.Draw()

# Classificazione k2>=3 && k3<=2 (VR012)

In [66]:
file_clust = r.TFile("Cluster_GSI1_cuts.root", "READ")

cluster_v1 = file_clust.Get("cluster_v1")

file_m = r.TFile("Clustering_man.root", "READ")

man_c = file_m.Get("man_tp")
cluster_v1.AddFriend(man_c)

In [67]:
file_pca = r.TFile("GSI1_PCA.root", "RECREATE")

campione_pca = cluster_v1.CopyTree("k1 >=1 && k2>= 2")
campione_pca.Write("pca")

file_pca.Close()

In [68]:
file_clust.Close()
file_m.Close()

In [69]:
principal = r.TPrincipal(3, "ND")

file_pca = r.TFile("GSI1_PCA.root", "READ")
info_pca = file_pca.Get("pca")

principal = r.TPrincipal(3, "ND")

for track in info_pca:
    vr0, vr1, vr2 = track.VR0_av, track.VR1_av, track.VR2_av
    vrs = np.zeros(3)
    vrs[0] = vr0
    vrs[1] = vr1
    vrs[2] = vr2
    principal.AddRow(vrs)
    
principal.MakePrincipals()
principal.MakeCode()

Writing on file "pca.C" ... done


In [70]:
r.gSystem.Load("pca_C.so")
vr123_tris = []

for track in info_pca:
    vr2, vr3, vr0 = track.VR0_av, track.VR1_av, track.VR2_av
    vrs = np.zeros(3)
    vrs[0] = vr2
    vrs[1] = vr3
    vrs[2] = vr0
    princ = np.zeros(3)
    principal.X2P(vrs, princ)
    vr123_tris.append(princ)

vr123_vr2_check = []
for i in vr123_tris:
    vr123_vr2_check.append(i[0])

In [71]:
file_2n = r.TFile("check_k3_2.root", "RECREATE")

pca_3 = r.TNtuple("pca_vr3_check", "vb", "VR012")

for i in range(len(vr123_vr2_check)):
    pca_3.Fill(vr123_vr2_check[i])
    
pca_3.Write()
file_2n.Close()

In [72]:
file_2n = r.TFile("check_k3_2.root", "READ")
pca_3 = file_2n.Get("pca_vr3_check")

check = r.TCanvas()
pca_3.Draw("VR012>>vr012_check_histo")
h82 = r.gDirectory.Get("vr012_check_histo")
h82.Draw("PE")
check.Draw()

In [73]:
fit_func2 = r.TF1("fit_func2", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -3.5, 5.5)

fit_func2.SetParameters(120, -1.5, .5, 60, 0.8, .7, 20, 3.5, 0.3)

fit_func2.SetParLimits(0, 90, 400)
fit_func2.SetParLimits(3, 30, 200)
fit_func2.SetParLimits(6, 10, 35)

fit_func2.SetParLimits(8, 0.001, .9)

In [74]:
r.Math.MinimizerOptions.SetDefaultMinimizer("Minuit")
r.Math.MinimizerOptions.SetDefaultTolerance(1e-10)
r.Math.MinimizerOptions.SetDefaultMaxIterations(200)

In [75]:
pca_3.Draw("VR012>>v123s(75, -4., 7.)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VP_{012} [k_{1}>=1 && k_{2}>=2]; VP_{012}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(120, -1.5, .5, 60, 0.9, 0.7, 40, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 400, 650)
fit_func.SetParLimits(3, 100, 350)
fit_func.SetParLimits(6, 100, 250)

#punto medio
fit_func.SetParLimits(1, -1.4, -1.)
fit_func.SetParLimits(4, -1, 2.)
fit_func.SetParLimits(7, 2., 4.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.5)
fit_func.SetParLimits(5, 0.2, 2)
fit_func.SetParLimits(8, 0.3, 2.5)

tr = -1.65

histos.Fit("fit_func", "S", "", tr, 7.)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "" )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 4)), "")
legend.Draw("SAME")

t1 = r.TText(4, 200, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")


c.Draw()

 FCN=59.8975 FROM MIGRAD    STATUS=CONVERGED     637 CALLS         638 TOTAL
                     EDM=2.54031e-11    STRATEGY= 1  ERROR MATRIX UNCERTAINTY   0.6 per cent
  EXT PARAMETER                                   STEP         FIRST   
  NO.   NAME      VALUE            ERROR          SIZE      DERIVATIVE 
   1  p0           4.12959e+02   3.19391e+01  -0.00000e+00  -1.31413e-05
   2  p1          -1.13903e+00   1.70984e-02   0.00000e+00  -2.52008e-05
   3  p2           4.27227e-01   3.42750e-02  -0.00000e+00  -1.58404e-06
   4  p3           2.61654e+02   1.55219e+01   0.00000e+00  -1.58339e-05
   5  p4          -9.03077e-02   1.53637e-01  -0.00000e+00   1.03222e-04
   6  p5           1.25834e+00   7.57461e-02   0.00000e+00  -3.52476e-05
   7  p6           1.77090e+02   7.58051e+00   0.00000e+00  -3.36606e-05
   8  p7           2.60539e+00   2.13435e-02   0.00000e+00  -3.22477e-04
   9  p8           3.69923e-01   1.75954e-02   0.00000e+00  -1.87639e-04


Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 


In [76]:
cn1, cn2, cn3 = [], [], []

file_pca2 = r.TFile("GSI1_PCA.root", "UPDATE")

cn1_t, cn2_t, cn3_t = r.TNtuple("cn1_t_II", "", "cl12"), r.TNtuple("cn2_t_III", "", "cl22"), r.TNtuple("cn3_t_IV", "", "cl32")

for value in vr123_vr2_check:
    if (value < tr):
        cn1.append(value)
        cn1_t.Fill(1)
        cn2_t.Fill(0)
        cn3_t.Fill(0)
    else:
        random_number = np.random.uniform(0,1)
        p1, p2, p3 = comp1.Eval(value)/fit_func.Eval(value), comp2.Eval(value)/fit_func.Eval(value), comp3.Eval(value)/fit_func.Eval(value)
        p_s = [p1, p2, p3]
        o_ps = sorted(p_s)
        id_1, id_2, id_3 = o_ps.index(p1), o_ps.index(p2), o_ps.index(p3)
    
        if (random_number <= o_ps[0]):
            if (id_1 == 0 and id_2!=0 and id_3!=0):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 0 and id_2==0 and id_3 !=0):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 0 and id_2!=0 and id_3 ==0):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
            
        elif(random_number > o_ps[0] and random_number <= o_ps[1]+o_ps[0]):
        
            if (id_1 == 1 and id_2!=1 and id_3!=1):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 1 and id_2==1 and id_3 !=1):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 1 and id_2!=1 and id_3 ==1):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
            
        elif(random_number > o_ps[1]+o_ps[0]):
            if (id_1 == 2 and id_2!=2 and id_3!=2):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 2 and id_2==2 and id_3 !=2):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 2 and id_2!=2 and id_3 ==2):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
    

In [77]:

cn1_t.Write("cnItr")
cn2_t.Write("cnIItr")
cn3_t.Write("cnIIItr")
file_pca2.Close()

In [78]:
file_pca = r.TFile("GSI1_PCA.root", "READ")

pca = file_pca.Get("pca")
cn1_tr = file_pca.Get("cnItr")
cn2_tr = file_pca.Get("cnIItr")
cn3_tr = file_pca.Get("cnIIItr")

pca.AddFriend(cn1_tr)
pca.AddFriend(cn2_tr)
pca.AddFriend(cn3_tr)


file4 = r.TFile("Z_GSI1_III.root", "RECREATE")

Z2_3_trk = pca.CopyTree("cl12==1 && cl22 == 0 && cl32 == 0 && k3<1")
Z2_3_trk.Write("Z2_3_trk")

z2_3, z3_3, z4_3 = [], [], []

for track in Z2_3_trk:
    z2_3.append(track.trid)

In [79]:
h4 = r.TH1F("h8", "", 100, -4, 6)
h5 = r.TH1F("h9", "", 100, -4, 6)
h6 = r.TH1F("h15", "", 100, -4, 6)

for i in cn1:
    h4.Fill(i)
for i in cn2:
    h5.Fill(i)
for i in cn3:
    h6.Fill(i)
        
hs = r.THStack("hs", "")
hs.Add(h4)
hs.Add(h6)
hs.Add(h5)

sp_canvas = r.TCanvas()
hs.SetTitle("VR012 (3 Gaussian Fit); VR012; Conteggi")
hs.Draw("pfc nostack")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
legend.AddEntry(h4, "Entries: "+ str(h4.GetEntries()), "f")
legend.AddEntry(h5, "Entries: "+ str(h5.GetEntries()), "f")
legend.AddEntry(h6, "Entries: "+ str(h6.GetEntries()), "f")
legend.Draw("SAME")
sp_canvas.Draw()

In [80]:
file4.Close()

In [81]:
file5 = r.TFile("Z_GSI1_III_2.root", "RECREATE")

Z3_3_trk = pca.CopyTree("cl12==0 && cl22 == 1 && cl32 == 0 && k3<1")
Z3_3_trk.Write("Z3_3_trk")

for track in Z3_3_trk:
    z3_3.append(track.trid)
    
file5.Close()

In [82]:
file6 = r.TFile("Z_GSI1_III_3.root", "RECREATE")
   
Z4_3_trk = pca.CopyTree("cl12==0 && cl22 == 0 && cl32 == 1 && k3<1")

Z4_3_trk.Write("Z4_3_trk")
    
for track in Z4_3_trk:
    z4_3.append(track.trid)
    
file6.Close()

In [83]:
check = z2_3 + z3_3 + z4_3

s = 0

for e in check:
    if (check.count(e) != 1):
        s+=1

print(s)
print(len(z2_3))
print(len(z3_3))
print(len(z4_3))
print(len(z2_3)+len(z3_3)+len(z4_3))

0
177
61
4
242


In [84]:
file_pca.Close()

# Classificazione k2<=2 && k3 >=2 (VR013)

In [85]:
file_clust = r.TFile("Cluster_GSI1_cuts.root", "READ")

cluster_v1 = file_clust.Get("cluster_v1")

file_m = r.TFile("Clustering_man.root", "READ")

man_c = file_m.Get("man_tp")
cluster_v1.AddFriend(man_c)

In [86]:
file_pca = r.TFile("GSI1_PCA.root", "RECREATE")

campione_pca = cluster_v1.CopyTree("k3>=2")
campione_pca.Write("pca")

file_pca.Close()

In [87]:
file_clust.Close()
file_m.Close()

In [88]:
principal = r.TPrincipal(3, "ND")

file_pca = r.TFile("GSI1_PCA.root", "READ")
info_pca = file_pca.Get("pca")

principal = r.TPrincipal(3, "ND")

for track in info_pca:
    vr0, vr1, vr2 = track.VR0_av, track.VR1_av, track.VR3_av
    vrs = np.zeros(3)
    vrs[0] = vr0
    vrs[1] = vr1
    vrs[2] = vr2
    principal.AddRow(vrs)
    
principal.MakePrincipals()
principal.MakeCode()

Writing on file "pca.C" ... done


In [89]:
r.gSystem.Load("pca_C.so")
vr123_tris = []

for track in info_pca:
    vr2, vr3, vr0 = track.VR0_av, track.VR1_av, track.VR3_av
    vrs = np.zeros(3)
    vrs[0] = vr2
    vrs[1] = vr3
    vrs[2] = vr0
    princ = np.zeros(3)
    principal.X2P(vrs, princ)
    vr123_tris.append(princ)

vr123_vr2_check = []
for i in vr123_tris:
    vr123_vr2_check.append(i[0])

In [90]:
file_2n = r.TFile("check_k3_2.root", "RECREATE")

pca_3 = r.TNtuple("pca_vr3_check", "vb", "VR013")

for i in range(len(vr123_vr2_check)):
    pca_3.Fill(vr123_vr2_check[i])
    
pca_3.Write()
file_2n.Close()

In [91]:
file_2n = r.TFile("check_k3_2.root", "READ")
pca_3 = file_2n.Get("pca_vr3_check")

check = r.TCanvas()
pca_3.Draw("VR013>>vr012_check_histo")
h82 = r.gDirectory.Get("vr012_check_histo")
check.Draw()

In [92]:
pca_3.Draw("VR013>>v123s(75, -4., 7.)", "", "COLZ")

histos = r.gDirectory.Get("v123s")
histos.SetTitle("VR013; VR013; Conteggi")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(120, -1.5, .5, 60, 0.9, 0.7, 40, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 400, 650)
fit_func.SetParLimits(3, 100, 350)
fit_func.SetParLimits(6, 100, 250)

#punto medio
fit_func.SetParLimits(1, -2.5, -1.)
fit_func.SetParLimits(4, -1, 2.)
fit_func.SetParLimits(7, 2., 4.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.5)
fit_func.SetParLimits(5, 0.2, 2)
fit_func.SetParLimits(8, 0.3, 2.5)

tr = -1.653

histos.Fit("fit_func", "S", "", tr, 7.)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(6)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()))
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()) )
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)))
legend.Draw("SAME")

c.Draw()

 FCN=45.4598 FROM MIGRAD    STATUS=CONVERGED     924 CALLS         925 TOTAL
                     EDM=4.94723e-13    STRATEGY= 1  ERROR MATRIX UNCERTAINTY   0.9 per cent
  EXT PARAMETER                                   STEP         FIRST   
  NO.   NAME      VALUE            ERROR          SIZE      DERIVATIVE 
   1  p0           4.21821e+02   3.61224e+01   0.00000e+00   1.30472e-06
   2  p1          -1.17621e+00   1.96692e-02   0.00000e+00  -7.30991e-06
   3  p2           4.63873e-01   4.42937e-02  -0.00000e+00  -1.10028e-05
   4  p3           2.44325e+02   1.71492e+01  -0.00000e+00  -9.65247e-07
   5  p4          -1.72146e-03   1.81479e-01  -0.00000e+00  -1.21182e-05
   6  p5           1.20428e+00   9.02197e-02   0.00000e+00  -3.01161e-05
   7  p6           1.60406e+02   7.21696e+00   0.00000e+00  -1.34676e-05
   8  p7           2.53030e+00   2.65075e-02  -0.00000e+00   2.89893e-05
   9  p8           4.27393e-01   2.01241e-02  -0.00000e+00  -9.19376e-06


Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 


In [93]:
cn1, cn2, cn3 = [], [], []

file_pca2 = r.TFile("GSI1_PCA.root", "UPDATE")

cn1_t, cn2_t, cn3_t = r.TNtuple("cn1_t_II2", "", "cl122"), r.TNtuple("cn2_t_III2", "", "cl222"), r.TNtuple("cn3_t_IV2", "", "cl322")

for value in vr123_vr2_check:
    if (value < tr):
        cn1.append(value)
        cn1_t.Fill(1)
        cn2_t.Fill(0)
        cn3_t.Fill(0)
    else:
        random_number = np.random.uniform(0,1)
        p1, p2, p3 = comp1.Eval(value)/fit_func.Eval(value), comp2.Eval(value)/fit_func.Eval(value), comp3.Eval(value)/fit_func.Eval(value)
        p_s = [p1, p2, p3]
        o_ps = sorted(p_s)
        id_1, id_2, id_3 = o_ps.index(p1), o_ps.index(p2), o_ps.index(p3)
    
        if (random_number <= o_ps[0]):
            if (id_1 == 0 and id_2!=0 and id_3!=0):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 0 and id_2==0 and id_3 !=0):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 0 and id_2!=0 and id_3 ==0):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
            
        elif(random_number > o_ps[0] and random_number <= o_ps[1]+o_ps[0]):
        
            if (id_1 == 1 and id_2!=1 and id_3!=1):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 1 and id_2==1 and id_3 !=1):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 1 and id_2!=1 and id_3 ==1):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
            
        elif(random_number > o_ps[1]+o_ps[0]):
            if (id_1 == 2 and id_2!=2 and id_3!=2):
                cn1.append(value)
                cn1_t.Fill(1)
                cn2_t.Fill(0)
                cn3_t.Fill(0)
            
            elif(id_1 != 2 and id_2==2 and id_3 !=2):
                cn2.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(1)
                cn3_t.Fill(0)
            
            elif(id_1 != 2 and id_2!=2 and id_3 ==2):
                cn3.append(value)
                cn1_t.Fill(0)
                cn2_t.Fill(0)
                cn3_t.Fill(1)
    

In [94]:
cn1_t.Write("cnItr2")
cn2_t.Write("cnIItr2")
cn3_t.Write("cnIIItr2")
file_pca2.Close()

In [95]:
file_pca = r.TFile("GSI1_PCA.root", "READ")

pca = file_pca.Get("pca")
cn1_tr = file_pca.Get("cnItr2")
cn2_tr = file_pca.Get("cnIItr2")
cn3_tr = file_pca.Get("cnIIItr2")

pca.AddFriend(cn1_tr)
pca.AddFriend(cn2_tr)
pca.AddFriend(cn3_tr)


file4 = r.TFile("Z_GSI1_IV.root", "RECREATE")

Z2_4_trk = pca.CopyTree("cl122==1 && cl222 == 0 && cl322 == 0 && k2<1")
Z2_4_trk.Write("Z2_4_trk")

z2_4, z3_4, z4_4 = [], [], []

for track in Z2_4_trk:
    z2_4.append(track.trid)

file4.Close()

In [96]:
file5 = r.TFile("Z_GSI1_IV_2.root", "RECREATE")

Z3_4_trk = pca.CopyTree("cl122==0 && cl222 == 1 && cl322 == 0 && k2<1")
Z3_4_trk.Write("Z3_4_trk")

for track in Z3_4_trk:
    z3_4.append(track.trid)
    
file5.Close()

In [97]:
file6 = r.TFile("Z_GSI1_IV_3.root", "RECREATE")
   
Z4_4_trk = pca.CopyTree("cl122==0 && cl222 == 0 && cl322 == 1 && k2<1")

Z4_4_trk.Write("Z4_4_trk")
    
for track in Z4_4_trk:
    z4_4.append(track.trid)
    
file6.Close()

In [98]:
check = z2_4 + z3_4 + z4_4

s = 0

for e in check:
    if (check.count(e) != 1):
        s+=1

print(s)
print(len(z2_4))
print(len(z3_4))
print(len(z4_4))
print(len(z2_4)+len(z3_4)+len(z4_4))

0
93
26
4
123


In [99]:
file_pca.Close()

# Assegnazione Carica

In [100]:
def GetVariables(tree):
    
    trids, tan_vr0s = [], []
    k_s = []
    
    for track in tree:
        trids.append(track.trid)
        tan_vr0s.append([track.tan, track.VR0_av,track.VR1_av, track.VR2_av, track.VR3_av] )
        k_s.append([track.k0, track.k1, track.k2, track.k3])
        
    return trids, tan_vr0s, k_s

In [101]:
file_2 = r.TFile(file_name, "READ")

tracks = file_2.Get("tracks_n")

trids, tan_vr0s = [], []
k_s = []

tr_ids, vol_s, ki_s, Zs = [], [], [], []

for track in tracks:
    trids.append(track.trid)
    tan_vr0s.append([track.tan, track.VR0_av,track.VR1_av, track.VR2_av, track.VR3_av] )
    k_s.append([track.k0, track.k1, track.k2, track.k3])
    
file_2.Close()
print(len(trids))

127879


In [102]:
####### primo file

file = r.TFile("Z_GSI1_I.root", "READ")

Z1_lE_trk = file.Get("Z1_lE_trk")

a, b, c = GetVariables(Z1_lE_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(1)
file.Close()

In [103]:
file2 = r.TFile("Z_GSI1_I.root", "READ")

Z1_hE_trk = file2.Get("Z1_hE_trk")

a, b, c = GetVariables(Z1_hE_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(1)
file2.Close()

In [104]:
file = r.TFile("Z_GSI1_I.root", "READ")

Z2_hE_trk = file.Get("Z2_hE_trk")

a, b, c = GetVariables(Z2_hE_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(2)

file.Close()

In [105]:
file2 = r.TFile("Z_GSI1_II.root", "READ")

Z2_2_trk = file2.Get("Z2_2_trk")

a, b, c = GetVariables(Z2_2_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(2)

file2.Close()

In [106]:
file2 = r.TFile("Z_GSI1_II_2.root", "READ")

Z3_2_trk = file2.Get("Z3_2_trk")

a, b, c = GetVariables(Z3_2_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(3)

file2.Close()

In [107]:
file2 = r.TFile("Z_GSI1_II_3.root", "READ")

Z4_2_trk = file2.Get("Z4_2_trk")

a, b, c = GetVariables(Z4_2_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(4)

file2.Close()

In [108]:
file3 = r.TFile("Z_GSI1_III.root", "READ")

Z2_3_trk = file3.Get("Z2_3_trk")

a, b, c = GetVariables(Z2_3_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(2)

file3.Close()

In [109]:
file3 = r.TFile("Z_GSI1_III_2.root", "READ")

Z3_3_trk = file3.Get("Z3_3_trk")
Zs.append(3)

a, b, c = GetVariables(Z3_3_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 

file3.Close()

In [110]:
file3 = r.TFile("Z_GSI1_III_3.root", "READ")

Z4_3_trk = file3.Get("Z4_3_trk")
Zs.append(4)

a, b, c = GetVariables(Z4_3_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 

file3.Close()

In [111]:
file3 = r.TFile("Z_GSI1_IV.root", "READ")

Z2_4_trk = file3.Get("Z2_4_trk")

a, b, c = GetVariables(Z2_4_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(2)

file3.Close()

In [112]:
file3 = r.TFile("Z_GSI1_IV_2.root", "READ")

Z3_4_trk = file3.Get("Z3_4_trk")

a, b, c = GetVariables(Z3_4_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(3)

file3.Close()

In [113]:
file3 = r.TFile("Z_GSI1_IV_3.root", "READ")

Z4_4_trk = file3.Get("Z4_4_trk")

a, b, c = GetVariables(Z4_4_trk)
tr_ids.append(a), vol_s.append(b), ki_s.append(c) 
Zs.append(4)

file3.Close()

eventualmente le tracce classificate con vr013

In [114]:
# check liste

L = 0

for i in range(len(tr_ids)):
    L += len(tr_ids[i])
    
print(L)
print(len(tr_ids))
print(len(Zs))

33058
12
12


In [115]:
## Definizione tree in uscita

file_2 = r.TFile(file_name, "READ")

tracks = file_2.Get("tracks_n")
Z_rec = np.zeros(1, dtype = np.intc)

output_file = r.TFile("output_cuts_GSI1.root", "RECREATE")

output_tree = tracks.CloneTree(0)
output_tree.Branch("Z", Z_rec, "Z/I")

In [116]:
def Assign_Charge(trids_to_assign, tan_vrs2, k_s2, Output_tree, Z):
   
    for i in range(len(trids_to_assign)):
        
        tracks.GetEntry(trids.index(trids_to_assign[i]))
        idx = trids.index(trids_to_assign[i])
        
        for j in range(len(tan_vr0s[idx])):
            if (tan_vr0s[idx][j] != tan_vrs2[i][j]):
                print("Errore")
                break
        for j in range(len(k_s[idx])):
            if (k_s[idx][j] != k_s2[i][j]):
                print("Errore")
                break
    
        Z_rec[0] = Z
        Output_tree.Fill()
        

In [117]:
## Assegnazione Carica per le tracce che rientrano nell'analisi
for i in range(len(tr_ids)):
    a, b, c = tr_ids[i], vol_s[i], ki_s[i]
    if (a != []):
        Assign_Charge(a, b, c,output_tree, Zs[i])

In [118]:
assigned_trids = []

for j in range(len(tr_ids)):
    for s in range(len(tr_ids[j])):
        assigned_trids.append(tr_ids[j][s])

In [119]:
len(assigned_trids)

33058

In [120]:
no_assigned_trids = trids.copy()

for el in assigned_trids:
    no_assigned_trids.remove(el)
    
print(len(list(set(no_assigned_trids).intersection(assigned_trids))))
print(len(no_assigned_trids))

0
94821


In [121]:
## Assegnazione Z=10 per le cariche che non rientrano nell'analisi

for i in range(len(no_assigned_trids)):
    tracks.GetEntry(trids.index(no_assigned_trids[i]))
    Z_rec[0] = 10
    output_tree.Fill()

In [122]:
## Scrittura Tree in Uscita

output_tree.Write("output_tree")

24934

In [123]:
z1_hE = tr_ids[0]
z1_lE = tr_ids[1]
z2_hE = tr_ids[2]
z2_2 = tr_ids[3]
z3_2 = tr_ids[4]
z4_2 = tr_ids[5]
z2_3 = tr_ids[6]
z3_3 = tr_ids[7]
z4_3 = tr_ids[8]
N_tot = 0

for i in range(len(tr_ids)):
    N_tot += len(tr_ids[i])

In [124]:
print("La percentuale di frammenti identificati come Z = 1 è: " + str( round(100*(len(z1_hE) + len(z1_lE))/(N_tot), 1  )) + " %")

La percentuale di frammenti identificati come Z = 1 è: 67.6 %


In [125]:
print("La percentuale di frammenti identificati come Z = 2 è: " + str( round(100*(len(z2_hE) + len(z2_2)+len(z2_3))/(N_tot), 1  )) + " %")

La percentuale di frammenti identificati come Z = 2 è: 20.0 %


In [126]:
print("La percentuale di frammenti identificati come Z = 3 è: " + str( round(100*(len(z3_3)+len(z3_2))/(N_tot), 1  )) + " %")

La percentuale di frammenti identificati come Z = 3 è: 8.9 %


In [127]:
print("La percentuale di frammenti identificati come Z >= 4 è: " + str( round(100*(len(z4_3)+len(z4_2) )/(N_tot), 1  )) + " %")

La percentuale di frammenti identificati come Z >= 4 è: 3.1 %


In [128]:
c = r.TCanvas()
output_tree.Draw("tan>>noZ", "Z == 10")

hs = r.THStack("hsss", "")

h1 = r.gDirectory.Get("noZ")
hs.Add(h1)

output_tree.Draw("tan>>Z", "Z != 10")
h1 = r.gDirectory.Get("Z")
hs.Add(h1)

hs.SetTitle("Distribuzione Angolare; tan(#theta); Conteggi")
hs.Draw("pfc nostack")

c.Draw()

In [129]:
c = r.TCanvas()

output_tree.Draw("tan>>Z1", "Z == 1")
hs = r.THStack("h4s", "")
h1 = r.gDirectory.Get("Z1")
hs.Add(h1)

output_tree.Draw("tan>>Z2", "Z == 2")
h2 = r.gDirectory.Get("Z2")
hs.Add(h2)

output_tree.Draw("tan>>Z3", "Z == 3")
h3 = r.gDirectory.Get("Z3")
hs.Add(h3)

output_tree.Draw("tan>>Z4", "Z == 4")
h4 = r.gDirectory.Get("Z4")
hs.Add(h4)

hs.SetTitle("Fragments Angular Distribution GSI1 (k_{0}>=1); tan(#theta); Entries")
hs.Draw("pfc nostack")

legend = r.TLegend(0.55,0.6,0.9,0.9)
legend.SetTextFont(0)
legend.SetTextSize(0.1)

def lazy(histo):
    return "Entries = "+str(histo.GetEntries())+"}{Mean: "+ str(round(histo.GetMean(), 2)) + ", RMS = "+str(round(histo.GetRMS(),2))+"}"

legend.AddEntry(h1, "#splitline{Z=1, " + lazy(h1))
legend.AddEntry(h2, "#splitline{Z=2, "+ lazy(h2))
legend.AddEntry(h3, "#splitline{Z=3, "+ lazy(h3))
legend.AddEntry(h4, "#splitline{Z>=4, "+ lazy(h4))
legend.Draw("SAME")

c.Draw()

In [130]:
output_tree.Draw("VR3_av", "Z != 10")

c.Draw()

In [ ]:
c = r.TCanvas()
output_tree.Draw("tan", "Z==10 && VR0_av < " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan))")
c.Draw()

In [ ]:
c = r.TCanvas()
output_tree.Draw("VR0_av", "tan<0.1 && Z==10 && VR1_av>0")
c.Draw()

In [ ]:
h1.SetFillColor(0)
h1.SetLineColor(1)
h1.SetLineWidth(2)

h2.SetFillColor(0)
h2.SetLineColor(4)
h2.SetLineWidth(2)

h3.SetFillColor(0)
h3.SetLineColor(95)
h3.SetLineWidth(2)

h4.SetFillColor(0)
h4.SetLineColor(8)
h4.SetLineWidth(2)


hs2 = r.THStack("hs2", "hs2")
hs2.SetTitle("Fragments Angular Distribution [k_{0}>=1];tan(#theta);Entries")

hs2.Add(h1)
hs2.Add(h2)
hs2.Add(h3)
hs2.Add(h4)

c = r.TCanvas()
hs2.Draw("nostack")

legend = r.TLegend(0.55,0.6,0.9,0.9)
legend.SetTextFont(0)
legend.SetTextSize(0.1)

def lazy(histo):
    return "Entries = "+str(histo.GetEntries())+"}{Mean: "+ str(round(histo.GetMean(), 2)) + ", RMS = "+str(round(histo.GetRMS(),2))+"}"

legend.AddEntry(h1, "#splitline{Z=1, " + lazy(h1))
legend.AddEntry(h2, "#splitline{Z=2, "+ lazy(h2))
legend.AddEntry(h3, "#splitline{Z=3, "+ lazy(h3))
legend.AddEntry(h4, "#splitline{Z>=4, "+ lazy(h4))
legend.Draw("SAME")

t1 = r.TText(0.5, 200, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")


c.Draw()

# Check Tabella

In [ ]:
c = r.TCanvas()
output_tree.Draw("tan>>k", "k0>0")

k = r.gDirectory.Get("k")
print(k.GetEntries())

c.Draw()

In [ ]:
c = r.TCanvas()
output_tree.Draw("tan>>k1", "VR0_av < " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k0>0 && k2<2 && k3<2 && k1<2")

k1 = r.gDirectory.Get("k1")
print(k.GetEntries())

c.Draw()

In [ ]:
c = r.TCanvas()
output_tree.Draw("tan>>k", "k0>0 && Z>0 && Z<5")

k = r.gDirectory.Get("k")
print(k.GetEntries())

c.Draw()

In [ ]:
kstack = r.THStack("ks", "ks")

k1.SetFillColor(0)
k1.SetLineColor(2)
k1.SetLineWidth(2)

k.SetFillColor(0)
k.SetLineColor(4)
k.SetLineWidth(2)

kstack.Add(k1)
kstack.Add(k)
kstack.SetTitle("Angular Distribution: Cosmic MIPs vs Fragments;tan(#theta);Entries")
c = r.TCanvas()

kstack.Draw("nostack")
legend = r.TLegend(0.55,0.6,0.9,0.9)
legend.SetTextFont(0)
legend.SetTextSize(0.1)
legend.AddEntry(k, "#splitline{Z=1, " + lazy(h1))
legend.AddEntry(k1, "#splitline{Z=2, "+ lazy(h2))
legend.Draw("SAME")

t1 = r.TText(0.5, 200, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")


c.Draw()

In [ ]:
c = r.TCanvas()
output_tree.Draw("tan>>k", "k0>0 && Z==1 && k1<2 && k2<2 && k3<2")

k = r.gDirectory.Get("k")
print(k.GetEntries())

c.Draw()

In [ ]:
c = r.TCanvas()
output_tree.Draw("tan>>k", "VR0_av < " + str(a2) + "*(1 + TMath::Exp(" + str(b2) + " *tan*tan)) && k0>0 && k3>1")

k = r.gDirectory.Get("k")
print(k.GetEntries())

c.Draw()

In [ ]:
f0 = r.TFile(track_name, "READ")

tracks = f0.Get("tracks")

In [ ]:
c = r.TCanvas()
tracks.Draw("s[0].Plate()>>h1", "", "")

h1 = r.gDirectory.Get("h1")
h1.SetFillColor(0)
h1.SetLineColor(4)
h1.SetLineWidth(2)

tracks.Draw("s[nseg-1].Plate()>>h2", "", "")
h2 = r.gDirectory.Get("h2")
h2.SetFillColor(0)
h2.SetLineColor(2)
h2.SetLineWidth(2)

hs = r.THStack("hs", "hs")
hs.Add(h1)
hs.Add(h2)
hs.SetTitle("First and Last Emulsion Film;Emulsion Film;Entries")
hs.Draw("nostack")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)

legend.AddEntry(h1, "Fist Segment")
legend.AddEntry(h2, "Last Segment")
legend.Draw("SAME")

t1 = r.TText(45, 30, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()

In [ ]:
c = r.TCanvas()

tracks.Draw("s.Plate()>>h0")
h0 = r.gDirectory.Get("h0")
h0.SetTitle("# Basetracks in emulsion films; Emulsion Film; Entries")
h0.Draw()

t1 = r.TText(45, 30, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()

In [ ]:
f0.Close()